In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — REPLAY DES 10 MATCHS : GÉNÉRATEUR DE CANDIDATS « VRAIES DÉTECTIONS + EPISODES » — 16/11/2026
# SEUL CHANGEMENT : la génération des candidats de coup d'envoi —
#   SCOUTIA_KO_PISTES=reelles (les pistes prédites ne comptent plus dans les signaux) + SCOUTIA_KO_DEDUP=episodes.
# Tout le reste = PRODUCTION (vérifié le 16/11 en comparant la cellule de replay et la cellule de production) :
# question Gemini, consigne « fenetre », fenêtre −25 s → +30 s, température 0, validation en résolution medium,
# classification low, escalade en résolution normale, second regard, réflexion réduite, anti-doublons.
# Heure de reprise injectée dans le replay (SCOUTIA_REPLAY_BORNES=1) pour appliquer les règles de la production.
# Attribution des équipes COUPÉE (non concernée ; économise ≈ 5,70 $).
# RÉFÉRENCE : run du 11/11 (production) sans ses 8 reprises de 2e mi-temps : 41 vrais buts sur 45, 6 faux buts dont
# 1 « confirmé » (Stembert 83:08). Manqués : penalties Spa 19:50, Spa 39:06, Raeren 69:12 ; Andrimont 84:06 (cache).
# CRITÈRES FIXÉS AVANT :
#   • les 3 penalties trouvés à ±40 s ;   • aucun des 41 buts du 11/11 perdu (±25 s, ou ±40 s signalé « heure ») ;
#   • faux buts « confirmés » ≤ 1 ;       • nouveaux faux buts : vérification vidéo ;   • coût mesuré.
# Vrais buts comptés à ±10, ±25 et ±40 s (un but trouvé à 40 s mais pas à 25 s = ERREUR D'HEURE, pas un faux but).
# ENTRÉES : codeprojectai, les 10 vidéos, mziparameter, test_resolution_gemini_cellule.py (version du 16/11).
# GitHub : analysis/detect_kickoff_candidates.py du 16/11 (SCOUTIA_KO_PISTES) + pipeline.py (SCOUTIA_REPLAY_BORNES).
# Pas de GPU. Save & Run All. COÛT ESTIMÉ ≈ 15-16 $ (non mesuré). ≈ 2 h 30 à 4 h.
# À GARDER : /kaggle/working/journaux_generateur.zip (journaux complets → traçabilité but par but).
# ════════════════════════════════════════════════════════════════════════════════════════════
import glob, os, re, subprocess, sys, zipfile

MATCHS = ["spa", "minerois_ster", "franchimont", "raeren", "andrimont", "p1_minerois", "goe", "stembert", "wanze", "juprelle"]
VERITE = {
 "raeren": ["16:39", "20:08", "27:18", "33:24", "45:25", "46:50", "50:17", "69:12", "91:00"],
 "andrimont": ["06:23", "30:31", "42:34", "48:58", "59:42", "84:06"],
 "franchimont": ["32:33", "49:51", "65:29", "79:42", "99:29"],
 "p1_minerois": ["16:58", "76:55", "110:53"], "goe": ["83:45", "114:29"], "stembert": ["20:27", "76:25", "110:45"],
 "wanze": ["22:50", "65:46"], "juprelle": ["22:50"],
 "minerois_ster": ["55:42", "87:16", "99:02", "103:02", "110:08"],
 "spa": ["19:50", "20:48", "39:06", "49:20", "64:10", "75:03", "89:05", "101:41", "112:45"]}
PENALTIES = {("spa", "19:50"), ("spa", "39:06"), ("raeren", "69:12")}
MANQUES_REF = {("spa", "19:50"), ("spa", "39:06"), ("raeren", "69:12"), ("andrimont", "84:06")}
FAUX_REF = {"minerois_ster": 1, "franchimont": 2, "raeren": 1, "goe": 1, "stembert": 1}     # 11/11 hors reprises
sec = lambda h: int(h.split(":")[0]) * 60 + int(h.split(":")[1])

# ── contrôles avant de payer ──────────────────────────────────────────────────────────────────
_src = next(iter(sorted(glob.glob("/kaggle/input/**/test_resolution_gemini_cellule.py", recursive=True))), None)
if not _src:
    raise SystemExit("❌ test_resolution_gemini_cellule.py introuvable (l'ajouter aux entrées) — rien n'est payé")
os.system("rm -rf /tmp/verif_depot && git clone -q --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git /tmp/verif_depot")
_dk = open("/tmp/verif_depot/analysis/detect_kickoff_candidates.py", encoding="utf-8").read() if os.path.exists("/tmp/verif_depot/analysis/detect_kickoff_candidates.py") else ""
_pp = open("/tmp/verif_depot/pipeline.py", encoding="utf-8").read() if os.path.exists("/tmp/verif_depot/pipeline.py") else ""
if "SCOUTIA_KO_PISTES" not in _dk or "SCOUTIA_REPLAY_BORNES" not in _pp:
    raise SystemExit("❌ GitHub : analysis/detect_kickoff_candidates.py (SCOUTIA_KO_PISTES) ou pipeline.py (SCOUTIA_REPLAY_BORNES) "
                     "du 16/11 absent — rien n'est payé")
BASE = open(_src, encoding="utf-8").read()
if '("DETR" not in x[0] or "DETR" in CACHE_REPLAY)' not in BASE:
    raise SystemExit("❌ test_resolution_gemini_cellule.py : ancienne version (sans la protection « DETR ») — rien n'est payé")
# on retire l'étape tenues + attribution (de « DÉTECTION DES TENUES » jusqu'au calcul du coût réel)
_a = BASE.index('print("\\n" + "=" * 60 + f"\\nDÉTECTION DES TENUES')
_b = BASE.index("# ── coût réel de ce run")
_VERIF_ATTR = '''
# ── 09/11 : vérification de l'ATTRIBUTION FAITE PAR LE PIPELINE (plus par le notebook) ──
import re as _re_v, collections as _col_v
_att_v = (result or {}).get("attribution_equipes") or {}
_ten_v = _att_v.get("tenues") or {}
def _parse_v(t):
    m = _re_v.match(r"maillot (.*) / short (.*)", str(t or ""))
    return (m.group(1), m.group(2)) if m else ("", "")
_TD_v = (_parse_v(_ten_v.get("A")), _parse_v(_ten_v.get("B")))
_mots_v = lambda x: set(str(x).lower().replace(",", " ").replace("/", " ").split()) - {"and", "with", "dark", "light"}
_sc_v = lambda det, vrai: len(_mots_v(det[0]) & _mots_v(vrai[0])) * 2 + len(_mots_v(det[1]) & _mots_v(vrai[1]))
_VT_v = VRAIES_TENUES_TOUTES[MATCH]; _VE_v = VERITE_EQUIPES[MATCH]
_d_v = _sc_v(_TD_v[0], _VT_v[0]) + _sc_v(_TD_v[1], _VT_v[1]); _i_v = _sc_v(_TD_v[0], _VT_v[1]) + _sc_v(_TD_v[1], _VT_v[0])
_map_v = None if _d_v == _i_v else ({"A": "A", "B": "B"} if _d_v > _i_v else {"A": "B", "B": "A"})
_bil_v = _col_v.Counter(); _vu_v = set()
for _b_v in (result or {}).get("buts", []):
    _g_v = float(_b_v.get("time_abs") or 0)
    _m_v = [h for h in _VE_v if abs(int(h.split(":")[0]) * 60 + int(h.split(":")[1]) - _g_v) <= 20 and h not in _vu_v]
    if not _m_v:
        continue
    _vu_v.add(_m_v[0])
    _e_v = _b_v.get("equipe")
    _ok_v = "—" if not _e_v else ("?" if not _map_v else ("juste" if _map_v[_e_v] == _VE_v[_m_v[0]] else "FAUX"))
    _bil_v[(_b_v.get("niveau_equipe") or "non attribué", _ok_v)] += 1
print(f"ATTRIBUTION PIPELINE : tenues {'OK' if _map_v else 'CORRESPONDANCE IMPOSSIBLE'} | "
      f"justes {_bil_v[('confirmé', 'juste')] + _bil_v[('probable', 'juste')]} | "
      f"FAUX {sum(v for k, v in _bil_v.items() if k[1] == 'FAUX')} | "
      f"non attribués {sum(v for k, v in _bil_v.items() if k[0] == 'non attribué')} | "
      f"coût {_att_v.get('cout_dollars', 0)} $ | escalades {_att_v.get('escalades', 0)} | "
      f"recompressées {_att_v.get('requetes_recompressees', 0)} | ÉCHECS {_att_v.get('questions_en_echec', 0)}")
'''
BASE = BASE[:_a] + "\n" + _VERIF_ATTR + "\n" + BASE[_b:]


def preparer(match):
    code = BASE
    for a, b in (('MATCH = "spa"     # ← seul réglage à changer', f'MATCH = "{match}"'),
                 ('KO_DEDUP = "fenetre"', 'KO_DEDUP = "episodes"'),
                 ('os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION',
                  'os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION\nos.environ["SCOUTIA_MEDIA_RESOLUTION_KO"] = "medium"\n'
                  'os.environ["SCOUTIA_MEDIA_RESOLUTION_CLASSIF"] = "low"\nos.environ["SCOUTIA_REPLAY_BORNES"] = "1"\n'
                  'os.environ["SCOUTIA_KO_PISTES"] = "reelles"\nos.environ["SCOUTIA_ATTRIBUTION"] = "0"')):
        if code.count(a) != 1:
            raise SystemExit(f"❌ réglage introuvable dans la cellule : {a!r} — rien n'est payé")
        code = code.replace(a, b)
    return code

def lancer(nom, code):
    p = f"/kaggle/working/run_{nom}.py"
    open(p, "w", encoding="utf-8").write(code)
    print(f"\n{'═' * 100}\n▶ {nom}\n{'═' * 100}", flush=True)
    proc = subprocess.Popen([sys.executable, p], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lignes = []
    for l in proc.stdout:
        lignes.append(l)
        if re.search(r"\[KO_DETECTION\] (déduplication|pistes|✅)|\[KO2\] REPLAY|but réel|but annoncé|BILAN DÉTECTION|"
                     r"GEMINI CACHE\] \d+ question|TOTAL ≈|Cache retenu|❌", l):
            print(l, end="", flush=True)
    proc.wait()
    sortie = "".join(lignes)
    open(f"/kaggle/working/journal_{nom}.txt", "w", encoding="utf-8").write(sortie)
    return sortie

BILAN = {}
for m in MATCHS:
    s = lancer(m, preparer(m))
    annonces = [(re.search(r"(\d+:\d+) — (\S+)", l.split("trouvé à")[1]).groups()) for l in re.findall(r"but réel .*✅ trouvé à.*", s)]
    annonces += re.findall(r"but annoncé (\d+:\d+) sans but réel — (\S+)", s)
    cout = re.search(r"TOTAL ≈ ([\d.]+) \$", s)
    pist = "pistes : vraies détections" in s
    ded = re.search(r"déduplication « (\w+) » : (\d+) candidats bruts → (\d+)", s)
    BILAN[m] = dict(annonces=[(sec(h), h, st) for h, st in annonces], cout=float(cout.group(1)) if cout else None,
                    reglages_ok=pist and bool(ded and ded.group(1) == "episodes"),
                    candidats=f"{ded.group(2)} → {ded.group(3)}" if ded else "?")

with zipfile.ZipFile("/kaggle/working/journaux_generateur.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob("/kaggle/working/journal_*.txt"):
        z.write(p, os.path.basename(p))

print(f"\n{'═' * 100}\nBILAN — générateur « vraies détections + episodes » (référence : 11/11, production)\n{'═' * 100}")
tot = {10: 0, 25: 0, 40: 0}; n_buts = 0; perdus = []; penal = []; faux_tot = 0; faux_conf = 0; cout_tot = 0.0; heure = []
for m in MATCHS:
    b = BILAN[m]; ann = list(b["annonces"]); libres = list(ann); res = {}
    for h in VERITE[m]:
        g = sec(h)
        prox = sorted(libres, key=lambda a: abs(a[0] - g))
        best = prox[0] if prox and abs(prox[0][0] - g) <= 40 else None
        if best:
            libres.remove(best)
        res[h] = (best[0] - g) if best else None
    n_buts += len(VERITE[m])
    for t in tot:
        tot[t] += sum(1 for v in res.values() if v is not None and abs(v) <= t)
    faux = [a for a in libres]
    faux_tot += len(faux); faux_conf += sum(1 for a in faux if a[2] == "confirmé")
    cout_tot += b["cout"] or 0
    for h, v in res.items():
        if (m, h) in PENALTIES:
            penal.append(f"{m} {h} : " + (f"trouvé ({v:+d} s)" if v is not None else "MANQUÉ"))
        if (m, h) not in MANQUES_REF and v is None:
            perdus.append(f"{m} {h}")
        if v is not None and 25 < abs(v) <= 40:
            heure.append(f"{m} {h} ({v:+d} s)")
    print(f"   {m:14s} réglages {'OK' if b['reglages_ok'] else '⚠️ NON APPLIQUÉS'} | candidats {b['candidats']:>12s} | "
          f"vrais {sum(1 for v in res.values() if v is not None and abs(v) <= 25)}/{len(VERITE[m])} (±25 s) | "
          f"faux {len(faux)} (11/11 : {FAUX_REF.get(m, 0)})" + (f" : {', '.join(a[1] + ' ' + a[2] for a in faux)}" if faux else "")
          + f" | {b['cout'] if b['cout'] is not None else '?'} $")
print(f"\n   vrais buts : ±10 s {tot[10]}/{n_buts} | ±25 s {tot[25]}/{n_buts} | ±40 s {tot[40]}/{n_buts}   (11/11 : 41/45 à ±25 s)")
print(f"   ERREURS D'HEURE (trouvés entre 25 et 40 s) : {', '.join(heure) or 'aucune'}")
print(f"   penalties : {' | '.join(penal)}")
print(f"   buts du 11/11 PERDUS : {', '.join(perdus) or 'aucun'}")
print(f"   faux buts : {faux_tot} dont {faux_conf} « confirmé(s) »   (11/11 hors reprises : 6 dont 1)")
ok = all("trouvé" in p for p in penal) and not perdus and faux_conf <= 1 and all(b["reglages_ok"] for b in BILAN.values())
print(f"   → {'CRITÈRES REMPLIS (reste : vérifier les nouveaux faux buts dans la vidéo)' if ok else 'CRITÈRES NON REMPLIS'}")
print(f"   COÛT RÉEL de la détection : ≈ {cout_tot:.2f} $")
print("   À envoyer : ce BILAN + /kaggle/working/journaux_generateur.zip (traçabilité but par but).")


════════════════════════════════════════════════════════════════════════════════════════════════════
▶ spa
════════════════════════════════════════════════════════════════════════════════════════════════════
  ✅ Cache retenu : /kaggle/input/datasets/michaelziant/codeprojectai/cache_spa_reference/meta.json (sauvegardé le 2026-10-19T10:38:38.480484)
  [KO2] REPLAY : reprise 63:39 | fin de 1re mi-temps 51:59 | source : trou du suivi dans le cache (11.7 min)
  [KO_DETECTION] pistes : vraies détections seulement (SCOUTIA_KO_PISTES=reelles)
  [KO_DETECTION] déduplication « episodes » : 1072 candidats bruts → 171
  [KO_DETECTION] ✅ GOAL confirmé à t=06:47 conf=0.95 | statut : à_vérifier (classification : NONE)
  [KO_DETECTION] ✅ GOAL confirmé à t=19:49 conf=0.95 | statut : à_vérifier (classification : NONE)
  [KO_DETECTION] ✅ GOAL confirmé à t=20:49 conf=0.98 | statut : à_vérifier (classification : NONE)
  [KO_DETECTION] ✅ GOAL confirmé à t=38:35 conf=0.95 | statut : à_vérifier (classificati